In [2]:
# 입력 예시 데이터
from scipy import stats
import numpy as np


#예시 값
logpress = [1, 1, 0, 1]    

streaming = [1, 0, 1, 0]
snapkv    = [1, 0, 0, 1]
pyramid   = [0, 1, 0, 1]
h2o       = [1, 1, 0, 0]

baseline = {"snapkv" : snapkv, "streaming" : streaming, "h2o" : h2o, "pyramid": pyramid}

# 원인 줄 메타데이터 (층 나누기용)
cause    = [2, 5, 7, 8]                  # 원인 줄 번호
is_error = {2: True, 5: False, 7: False, 8: True} # 2: 에러 맞음 5: 에러 아님 7: 에러 아님 8: 에러 맞음
pos      = {2: 0.30, 5: 0.55, 7: 0.80, 8: 0.90}

keep = {2: True, 5: False, 7: True, 8: True} # keep mask --> 해당 줄이 살아남았나 죽었나

overall = [0,1,2,3]
non_error = [i for i in range(4) if not is_error[cause[i]]]
middle = [i for i in range (4) if 0.25 <= pos[cause[i]] <= 0.75]

strata = {"overall" : overall, "non_error" : non_error, "middle" : middle}


In [3]:
def preservation(result, target): # 보존율 함수(결과, 타겟(압축기))
    return sum(1 for c in target if result[c]) / len(target) 
# target 안에 원소 c를 하나씩 꺼내서, 만약 c번째 인덱스 결과배열에서 값이 true(1)이라면 + 1 합계 구하기. / 타겟의 전체길이. 
# --> true가 전체 타겟에서 얼마나 남아있는가?의 비율.

print("-- 층별 보존율 -- ")
print(f"{'method':10}", *[f"{s:>10}" for s in strata])
for name, res in {"logpress" : logpress, **baseline}.items():
    rates = [preservation(res, t) for t in strata.values()]
    print(f"{name:10}", *[f"{r:>10.2f}" for r in rates])


-- 층별 보존율 -- 
method        overall  non_error     middle
logpress         0.75       0.50       1.00
snapkv           0.50       0.00       0.50
streaming        0.50       0.50       0.50
h2o              0.50       0.50       1.00
pyramid          0.50       0.50       0.50


In [ ]:
from math import comb

def binom_cdf_half(k,n):
    pmf = 0.5 ** n  # 동전을 n번 던지는데, 특정 결과 하나가 나올 확률
    cdf = pmf # k=0 확률부터 시작
    for i in range(1, k+1):
        pmf += (n - i + 1) / i # 다음항 계산.
        cdf += pmf # pmf의 누적
    return cdf

def mcnemar(x,y, idx): #치우친 결과가 우연히 나올 확률? -> 누적 확률
    p = q = 0
    for i in idx:
        if x[i] != y[i]:
            if x[i] == 1:
                p += 1
            else:
                q += 1
    n = p+q
    if n == 0: return p,q,1.0
    
    cdf = binom_cdf_half(min(p,q),n)
    p_value = min(2*cdf, 1.0) # 분포의 좌우대칭성 이용. 반쪽만 측정후, 2배. 
    return p,q, p_value


results = []
for b_name, base in baseline.items():
    for s_name, idx in strata.items():
        p,q,p_value = mcnemar(logpress, base, idx)
        results.append([b_name, s_name, p, q, p_value])
        print(f"{b_name:10} {s_name:10} p = {p} q = {q} p_value = {round(p_value,3)}")


In [ ]:
#p값 BH보정 후 Tier-1 pass/fail
#임의의 p값들
p_values = [0.001, 0.008, 0.02, 0.03, 0.04, 0.045,
         0.3, 0.4, 0.5, 0.6, 0.8, 0.9]

def bh_fdr(p_values, alpha = 0.05):
    p_values = np.array(p_values)
    m = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adj = ranked * m / (np.arange(m) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.clip(adj, 0, 1)
    return out

p_adj = bh_fdr([r[4] for r in results])

for r, pa in zip(results, p_adj):
    b_name, s_name, l_win, b_win, p_value = r
    if l_win > b_win:
        verdict = f"logpress 우세 ({l_win} vs {b_win})"
    elif b_win > l_win:
        verdict = f"{b_name} 우세 ({b_win} vs {l_win})"
    else:
        verdict = "동률"
    print(f"[{s_name:10}] logpress vs {b_name:10} -> {verdict}, p_bh={pa:.3f}")
    
hard = [(r, pa) for r, pa in zip(results, p_adj) if r[1]!= "overall"] 
passed = all(pa< 0.05 and r[2] > r[3] for r,pa in hard)
print("\nTier-1:", "Pass" if passed else "Fail(표본이 작으면 정상)")   

In [ ]:
#오늘 완성.... 고민거리를 토요일에 더 고민해보는걸로.... 그리고 설계 깔짝? 이걸 어떻게 해야 할수 있겠냐,..